# Subtitles Generator

Pipeline overview:
1. Load audio from `sample_data/`
2. Detect speech segments with Silero VAD
3. Translate each segment with Sarvam STT (`saaras:v3`, mode=`translate`)
4. Write `outputs/subtitles.srt`


In [ ]:
%pip install -r requirements.txt


## Setup


In [ ]:
from __future__ import annotations

import io
import os
from pathlib import Path

import librosa
import numpy as np
import torch
from dotenv import load_dotenv
from pydub import AudioSegment
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
SAMPLE_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 16000
VAD_THRESHOLD = 0.5
COMBINE_DURATION = 8.0
COMBINE_GAP = 1.0


## VAD helpers


In [ ]:
@torch.no_grad()
def get_vad_probs(model: torch.nn.Module, audio: np.ndarray, sample_rate: int = 16000) -> list[float]:
    audio_t = torch.as_tensor(audio, dtype=torch.float32)
    window = 512 if sample_rate == 16000 else 256
    model.reset_states()
    probs: list[float] = []
    for start in range(0, len(audio_t), window):
        chunk = audio_t[start : start + window]
        if len(chunk) < window:
            chunk = torch.nn.functional.pad(chunk, (0, int(window - len(chunk))))
        probs.append(float(model(chunk, sample_rate).item()))
    return probs


def get_utterances(vad_probs: list[float], threshold: float = 0.5, frame_duration: float = 0.032) -> list[tuple[float, float]]:
    utterances: list[tuple[float, float]] = []
    in_utt = False
    start = 0.0
    for i, prob in enumerate(vad_probs):
        if prob > threshold and not in_utt:
            in_utt = True
            start = i * frame_duration
        elif prob <= threshold and in_utt:
            in_utt = False
            end = i * frame_duration
            if end > start:
                utterances.append((start, end))
    if in_utt:
        utterances.append((start, len(vad_probs) * frame_duration))
    return utterances


def merge_segments(
    segments: list[tuple[float, float]],
    max_duration: float = 8.0,
    max_gap: float = 1.0,
) -> list[tuple[float, float]]:
    if not segments:
        return []
    merged: list[tuple[float, float]] = []
    cur_start, cur_end = segments[0]
    for start, end in segments[1:]:
        if (start - cur_end <= max_gap) and (end - cur_start <= max_duration):
            cur_end = end
        else:
            merged.append((cur_start, cur_end))
            cur_start, cur_end = start, end
    merged.append((cur_start, cur_end))
    return merged


## Transcription + SRT


In [ ]:
def detect_segments(audio_file: Path) -> list[tuple[float, float]]:
    vad_model, _ = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False,
        onnx=False,
    )
    vad_model.eval()
    audio, _ = librosa.load(str(audio_file), sr=SAMPLE_RATE)
    probs = get_vad_probs(vad_model, audio, SAMPLE_RATE)
    return merge_segments(
        get_utterances(probs, threshold=VAD_THRESHOLD),
        max_duration=COMBINE_DURATION,
        max_gap=COMBINE_GAP,
    )


def transcribe_segment(audio: AudioSegment, start_sec: float, end_sec: float) -> str:
    segment = audio[int(start_sec * 1000) : int(end_sec * 1000)]
    buf = io.BytesIO()
    segment.export(buf, format="wav")
    buf.seek(0)
    response = client.speech_to_text.transcribe(
        file=("segment.wav", buf, "audio/wav"),
        model="saaras:v3",
        mode="translate",
        language_code="unknown",
    )
    if hasattr(response, "transcript"):
        return response.transcript or ""
    if isinstance(response, dict):
        return response.get("transcript", "") or ""
    return str(response)


def format_timestamp(seconds: float) -> str:
    ms = int((seconds % 1) * 1000)
    total = int(seconds)
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"


def write_srt(results: list[dict], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as fh:
        for i, row in enumerate(results, start=1):
            fh.write(f"{i}\n")
            fh.write(
                f"{format_timestamp(row['start_time'])} --> {format_timestamp(row['end_time'])}\n"
            )
            fh.write(f"{row['transcript']}\n\n")


## Run


In [ ]:
AUDIO_PATH = SAMPLE_DIR / "clip.wav"
if not AUDIO_PATH.exists():
    raise FileNotFoundError(
        f"Missing {AUDIO_PATH}. Add an audio file under sample_data/ first."
    )

segments = detect_segments(AUDIO_PATH)
if not segments:
    raise RuntimeError(f"No speech segments detected in {AUDIO_PATH}")

audio = AudioSegment.from_file(AUDIO_PATH)
results: list[dict] = []
for start, end in segments:
    transcript = transcribe_segment(audio, start, end)
    if transcript:
        results.append({"start_time": start, "end_time": end, "transcript": transcript})
        print(f"[{start:.1f}-{end:.1f}] {transcript}")

srt_path = OUTPUT_DIR / "subtitles.srt"
write_srt(results, srt_path)
print(f"Wrote {srt_path}")
